# E5 — Real-trace external validity (tau2 → verifier labels)

Maps real tau2-bench agent trajectories (Apache-2.0) into our
(root, chain, action) schema; **our verifier assigns every label**
(authorized = the agent's real in-scope tool calls; unauthorized = the same
real calls redirected to a foreign customer's real IDs — a confused deputy
on real calls). Then evaluates the released CE model per domain, plus the
lexical-heuristic floor for honesty (the redirect construction is a
resource-scope task, so the floor is expected to be high — report it).

Evaluation-only (no training). Runtime → T4 GPU → Run all (~10 min).
Download `realtrace_results.zip` at the end.

In [ ]:
import os
os.chdir('/content')
if not os.path.isdir('verifier-as-reward'):
    !git clone https://github.com/esmaeil-abedi-dev/verifier-as-reward.git
os.chdir('/content/verifier-as-reward')
!git pull --ff-only
import torch; print('cuda:', torch.cuda.is_available())
!PYTHONPATH=. python test_authority_verifier.py

## Step 1 — map tau2 trajectories (downloads the dataset, ~30 s)

In [ ]:
!PYTHONPATH=. python -u map_tau_to_chain.py --seed 5
!PYTHONPATH=. python test_map_tau_to_chain.py

## Step 2 — released CE model on each domain + combined

In [ ]:
import json; json.dump({'backends': {}}, open('results_realtrace.json', 'w'))
!PYTHONPATH=. python train_verifier_reward.py --eval-checkpoint esmaeil-abedi-dev/verifier-ce-qwen2.5-0.5b --test-file real_trace_telecom.jsonl --merge-results results_realtrace.json
!PYTHONPATH=. python train_verifier_reward.py --eval-checkpoint esmaeil-abedi-dev/verifier-ce-qwen2.5-0.5b --test-file real_trace_airline.jsonl --merge-results results_realtrace.json
!PYTHONPATH=. python train_verifier_reward.py --eval-checkpoint esmaeil-abedi-dev/verifier-ce-qwen2.5-0.5b --test-file real_trace_retail.jsonl --merge-results results_realtrace.json
!PYTHONPATH=. python train_verifier_reward.py --eval-checkpoint esmaeil-abedi-dev/verifier-ce-qwen2.5-0.5b --test-file real_trace_all.jsonl --merge-results results_realtrace.json

## Step 3 — lexical-heuristic floor (honesty baseline) + summary with Wilson CIs

In [ ]:
import json
from eval_harness import run_eval, make_heuristic
from trace_benchmark import load_traces
from stats import wilson_ci
results = json.load(open('results_realtrace.json'))
for tag in ['telecom', 'airline', 'retail', 'all']:
    m = run_eval(make_heuristic(), load_traces(f'real_trace_{tag}.jsonl'))['metrics']
    results['backends'][f'heuristic::real_trace_{tag}.jsonl'] = {'metrics': m}
json.dump(results, open('results_realtrace.json', 'w'), indent=2)

print(f"{'backend/test':52} {'n':>4} {'acc% [CI]':>22} {'f-auth%':>8} {'f-ref%':>7}")
for name, b in results['backends'].items():
    m = b['metrics']; n = m['n_actions']
    c = wilson_ci(round(m['accuracy'] * n), n)
    fa = m['false_authorize_rate']; fr = m['false_refuse_rate']
    print(f"{name[:52]:52} {n:4d} {100*m['accuracy']:5.1f} [{100*c[0]:.1f},{100*c[1]:.1f}]"
          f" {'n/a' if fa is None else format(100*fa,'6.1f')} {'n/a' if fr is None else format(100*fr,'5.1f')}")

In [ ]:
!zip -j realtrace_results.zip results_realtrace.json real_trace_*.jsonl
from google.colab import files
files.download('realtrace_results.zip')